# BidMate Embedding LoRA Fine-tune — KURE-v1 on Korean RFP

Issue [#179](https://github.com/hskim-solv/BidMate-DocAgent/issues/179) · ADR [0025](../docs/adr/0027-lora-finetuned-embedding-additive.md) · Phase 3.4.

Reproduces the trained LoRA adapter that the `agentic_full_finetuned` /
`naive_baseline_finetuned` ablation rows in [`eval/config.yaml`](../eval/config.yaml)
consume. Designed for a free Colab T4 (12 GB VRAM) — full run ≈ 30 min.

**Inputs.** `data/training/embedding_pairs.jsonl` (regenerable from
`data/raw/` via [`scripts/generate_finetune_pairs.py`](../scripts/generate_finetune_pairs.py)).

**Output.** A PEFT LoRA adapter saved to `lora_adapter/` (≈ 30 MB) and pushed
to the Hugging Face Hub repo `bidmate/embedding-lora-kure-rfp-ko-v1`. The
adapter commit SHA is then pinned in `eval/config.yaml` under both new ablation
rows (ADR 0027 supply-chain rule).

**Important.** This notebook is committed with empty outputs by design — see
`tests/notebooks/test_embedding_finetune_executes.py` for the structural smoke
that runs in CI. The training results are filled into
[`docs/embedding-finetune.md`](../docs/embedding-finetune.md) in #179.

## 1. 환경 설치

In [ ]:
# Colab T4 기준 ~2분 소요. 로컬 GPU 환경이면 이미 설치된 패키지는 스킵됩니다.
%pip install -q \
    peft>=0.13,<1.0 \
    sentence-transformers>=2.7,<3.0 \
    datasets>=2.19 \
    accelerate>=0.30 \
    huggingface_hub>=0.23

## 2. GPU 확인

T4 미만 (예: K80) 에서는 batch 16으로 줄여야 OOM을 피합니다.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "This notebook requires a CUDA GPU. "
    "In Colab: Runtime → Change runtime type → T4 GPU."
)
print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"CUDA memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. 리포지토리 마운트

* Colab → `git clone` 으로 가져옴.
* 로컬에서 실행 시 (`!ls ../scripts/generate_finetune_pairs.py` 가 보임)
  → 기존 체크아웃 그대로 사용.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_NAME = "BidMate-DocAgent"
if Path("scripts/generate_finetune_pairs.py").exists():
    REPO_ROOT = Path.cwd()
elif Path("../scripts/generate_finetune_pairs.py").exists():
    REPO_ROOT = Path.cwd().parent
else:
    # Colab cold start
    if not Path(REPO_NAME).exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            f"https://github.com/hskim-solv/{REPO_NAME}.git",
        ])
    REPO_ROOT = Path(REPO_NAME).resolve()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print(f"REPO_ROOT = {REPO_ROOT}")

## 4. 학습 데이터 생성

`scripts/generate_finetune_pairs.py` 가
`data/training/embedding_pairs.jsonl` 을 생성합니다.

* **stub 백엔드** (기본, 무료) — 결정적 템플릿, 플러밍 검증용.
* **anthropic 백엔드** — `BIDMATE_PAIRGEN_API_KEY` 환경변수 + Claude 4.6
  Sonnet 사용. 실제 학습용 paraphrase 생성 (~$5 미만).

Eval 표면(`eval/dev_queries_v1.jsonl`, `eval/config.yaml`,
`eval/multiturn_scenarios_v1.jsonl`) 과 겹치는 쿼리는
스크립트가 자동으로 거부하며, 거부율이 5% 를 넘으면 fail-loud 종료.

In [ ]:
import os

# 실제 학습용은 anthropic 으로 변경:
#   os.environ["BIDMATE_PAIRGEN_BACKEND"] = "anthropic"
#   os.environ["BIDMATE_PAIRGEN_API_KEY"] = "sk-ant-..."  # 노트북에 하드코딩 금지
#   os.environ["BIDMATE_PAIRGEN_MODEL"]   = "claude-sonnet-4-6"

subprocess.check_call([
    sys.executable, "scripts/generate_finetune_pairs.py",
    "--queries_per_chunk", "200",
    "--neg_per_pos", "3",
    "--seed", "17",
    "--output", "data/training/embedding_pairs.jsonl",
])

## 5. 베이스 임베딩 로드 — `nlpai-lab/KURE-v1`

BGE-M3 대신 KURE-v1 을 베이스로 선택한 이유는 ADR 0027 §Decision 참조:

* 한국어 특화 (`nlpai-lab` 학습), 1.1 GB — T4 12 GB 헤드룸에 batch 32 가능.
* 대칭 encoder (BGE-M3 은 dense / sparse / multi-vector 비대칭 → LoRA target
  결정이 추가 ADR-worthy).
* Phase 1.2 (ADR 0019) 측정 표면에 이미 등장.

In [ ]:
from sentence_transformers import SentenceTransformer

BASE_MODEL = "nlpai-lab/KURE-v1"
model = SentenceTransformer(BASE_MODEL)
model = model.to("cuda")

total = sum(p.numel() for p in model.parameters())
print(f"Base params: {total/1e6:.1f} M")

## 6. LoRA 적용

* `r=16`, `alpha=32` — XLM-RoBERTa 기반 임베딩에 흔히 쓰이는 안정 조합.
* `target_modules = ["query","key","value","dense"]` — attention 4 슬롯 모두
  학습 (encoder 가 대칭이므로 single head 만 만지면 됨).
* `task_type="FEATURE_EXTRACTION"` — 분류 head 가 없는 임베딩이라는 표시.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["query", "key", "value", "dense"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION,
)
underlying = model[0].auto_model
peft_underlying = get_peft_model(underlying, lora_config)
model[0].auto_model = peft_underlying

trainable = sum(p.numel() for p in peft_underlying.parameters() if p.requires_grad)
total     = sum(p.numel() for p in peft_underlying.parameters())
print(f"LoRA trainable: {trainable/1e6:.2f} M ({100*trainable/total:.2f} % of {total/1e6:.1f} M)")

## 7. 학습 / 검증 데이터 로드

JSONL → `InputExample(texts=[query, positive_text])` 로 변환. 음성 샘플은
`MultipleNegativesRankingLoss` 가 in-batch negatives 로 활용하므로 별도
전달은 불필요. 다만 BM25-mined hard negatives 는 같은 batch 에 같이
넣어 contrastive signal 을 강화합니다.

In [ ]:
import json, random
from sentence_transformers import InputExample
from torch.utils.data import DataLoader

pairs_path = REPO_ROOT / "data" / "training" / "embedding_pairs.jsonl"
rows = [json.loads(line) for line in pairs_path.read_text(encoding="utf-8").splitlines() if line.strip()]

train_examples = []
val_pairs      = []
for r in rows:
    pair = InputExample(texts=[r["query"], r["positive_text"]])
    if r["split"] == "val":
        val_pairs.append((r["query"], r["positive_text"], r["positive_chunk_id"]))
    else:
        train_examples.append(pair)
        # hard negatives → extra contrastive examples (anchor=query, negative_text)
        for neg in r.get("negatives", []):
            train_examples.append(InputExample(texts=[r["query"], neg["text"]], label=0.0))

random.Random(17).shuffle(train_examples)
train_loader = DataLoader(train_examples, batch_size=32, shuffle=True)
print(f"train examples: {len(train_examples)}  ·  val pairs: {len(val_pairs)}")

## 8. 학습 전 베이스라인 측정 (val rank-1 / rank-5)

전체 코퍼스(sub-chunk) 풀을 후보로 두고 val query 가 자기 positive
chunk 를 top-K 로 잡는지 측정. 학습 후와 비교하기 위해 *지금* 한 번 찍어둠.

In [ ]:
import numpy as np
from rag_core import build_chunks, load_raw_documents

subchunks = build_chunks(load_raw_documents(REPO_ROOT / "data" / "raw"), max_chars=240, overlap_sentences=1)
corpus_texts = [c["text"] for c in subchunks]
corpus_ids   = [c["chunk_id"] for c in subchunks]

def measure_recall(model, val_pairs, corpus_texts, corpus_ids):
    corpus_vecs = model.encode(corpus_texts, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
    q_texts = [q for q, _, _ in val_pairs]
    q_vecs  = model.encode(q_texts, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
    scores  = q_vecs @ corpus_vecs.T
    ranks   = np.argsort(-scores, axis=1)
    hits1, hits5 = 0, 0
    for i, (_, _, gold_id) in enumerate(val_pairs):
        top5 = [corpus_ids[j] for j in ranks[i, :5]]
        if gold_id == top5[0]: hits1 += 1
        if gold_id in top5:    hits5 += 1
    return hits1 / max(1, len(val_pairs)), hits5 / max(1, len(val_pairs))

r1_base, r5_base = measure_recall(model, val_pairs, corpus_texts, corpus_ids)
print(f"BEFORE  rank@1 = {r1_base:.3f}   rank@5 = {r5_base:.3f}")

## 9. 학습 — `MultipleNegativesRankingLoss`

* 1 epoch, batch 32, lr 2e-5, cosine schedule, warmup 10 %.
* in-batch negatives: 각 anchor 의 나머지 batch 멤버가 자동 음성 — batch 32
  → effective 31 negatives 가 free 로 따라옴.
* T4 기준 약 20-30 분.

In [ ]:
from sentence_transformers import losses

train_loss = losses.MultipleNegativesRankingLoss(model=model)

model.fit(
    train_objectives=[(train_loader, train_loss)],
    epochs=1,
    warmup_steps=int(0.1 * len(train_loader)),
    optimizer_params={"lr": 2e-5},
    scheduler="warmupcosine",
    show_progress_bar=True,
    use_amp=True,
)

## 10. 학습 후 측정

In [ ]:
r1_after, r5_after = measure_recall(model, val_pairs, corpus_texts, corpus_ids)
print(f"AFTER   rank@1 = {r1_after:.3f}   rank@5 = {r5_after:.3f}")
print(f"Δ rank@1: {r1_after - r1_base:+.3f}   Δ rank@5: {r5_after - r5_base:+.3f}")

## 11. LoRA 어댑터 저장

베이스 모델은 저장하지 않음 — HF Hub 의 `nlpai-lab/KURE-v1` 을 그대로 참조.
`merge_and_unload()` 는 *추론 시* 적용 (ADR 0027 §Decision rule 1) — 저장은
PEFT 형태 그대로.

In [ ]:
ADAPTER_DIR = REPO_ROOT / "lora_adapter"
model[0].auto_model.save_pretrained(str(ADAPTER_DIR))
print(f"Saved adapter → {ADAPTER_DIR}")
import os
size_mb = sum(p.stat().st_size for p in ADAPTER_DIR.rglob("*") if p.is_file()) / 1e6
print(f"Adapter size:  {size_mb:.1f} MB")

## 12. Hugging Face Hub 업로드

토큰은 *절대* 노트북에 하드코딩하지 말 것. Colab Secrets 또는 환경변수로만 전달.

업로드 완료 후 반환되는 commit SHA 를 `eval/config.yaml` 의
`embedding_lora_adapter` 필드의 `<sha>` placeholder 자리에 박아 넣는 것이
ADR 0027 §Decision rule 3 (SHA pinning) 의 마지막 단계.

In [ ]:
from huggingface_hub import HfApi, login
import os

token = os.environ.get("HF_TOKEN")
assert token, (
    "HF_TOKEN is not set. In Colab: Secrets → add HF_TOKEN with write access. "
    "Locally: export HF_TOKEN=hf_... before launching jupyter."
)
login(token=token, add_to_git_credential=False)

HF_REPO_ID = "bidmate/embedding-lora-kure-rfp-ko-v1"
api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", private=False, exist_ok=True)
result = api.upload_folder(
    folder_path=str(ADAPTER_DIR),
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message=f"LoRA r=16 KURE-v1 on RFP pairs (val rank@1 {r1_after:.3f} ↑ {r1_after - r1_base:+.3f})",
)
print(f"Uploaded.  Commit SHA: {result.oid}")
print(f"Pin this SHA in eval/config.yaml under both:")
print(f"  - agentic_full_finetuned.embedding_lora_adapter   = {HF_REPO_ID}@{result.oid}")
print(f"  - naive_baseline_finetuned.embedding_lora_adapter = {HF_REPO_ID}@{result.oid}")

## 13. (마지막) 데이터 누수 재검증

학습 데이터에 eval 쿼리가 새어 들어가지 않았는지 노트북 안에서 한 번 더
확인. `scripts/generate_finetune_pairs.py` 가 이미 fail-loud 거부하지만,
PR4 PR template item 5b 에 첨부할 수치 (`rejected / generated`) 가 여기서
나옴.

In [ ]:
from scripts.generate_finetune_pairs import ContaminationGuard, load_eval_queries

guard = ContaminationGuard.from_eval_queries(load_eval_queries())
queries_in_train = [r["query"] for r in rows]
leaked = [q for q in queries_in_train if guard.is_contaminated(q)]
print(f"train queries: {len(queries_in_train)}")
print(f"leaked into eval surfaces: {len(leaked)}  (must be 0)")

---

## 후속 작업 (#179)

1. 위 출력의 commit SHA 를 `eval/config.yaml` 두 행에 박기.
2. `python scripts/run_embedding_ablation.py --models nlpai-lab/KURE-v1` 을
   `BIDMATE_EMBEDDING_LORA_ADAPTER` 켠/끈 상태에서 한 번씩 실행 → real-data
   delta 측정.
3. `docs/embedding-finetune.md` 의 결과 표 직접 채우기 (cognitive-ownership
   boundary — model card / rationale 은 본인 framing 으로).
4. README ablation 표에 두 행 추가, bootstrap CI 밴드 (`eval/bootstrap.py`)
   재계산.